In [1]:
import matplotlib.pyplot as plt
import numpy as np

In [2]:
from fetch_stac import fetch_s2
from utils import softclip

In [3]:
STAC_URL = "https://earth-search.aws.element84.com/v1"

In [4]:
BANDS = [
    "coastal",    # B1: Coastal Aerosol, 60 m/px, 443 nm, Atmospheric correction
    "blue",       # B2: Blue, 10 m/px, 490 nm, Forest type mapping
    "green",      # B3: Green, 10 m/px, 560 nm, Water body analysis
    "red",        # B4: Red, 10 m/px, 665 nm, Vegetation discrimination
    "rededge1",   # B5: Red Edge 1, 20 m/px, 705 nm, Vegetation stress detection
    "rededge2",   # B6: Red Edge 2, 20 m/px, 740 nm, Leaf area index monitoring
    "rededge3",   # B7: Red Edge 3, 20 m/px, 783 nm, Biomass estimation
    "nir",        # B8: NIR, 10 m/px, 842 nm, Shorelines and biomass
    "nir08",      # B8A: Narrow NIR, 20 m/px, 865 nm, Vegetation classification
    "nir09",      # B9: Water Vapour, 60 m/px, 945 nm, Atmospheric water vapour
    "swir16",     # B11: SWIR 1, 20 m/px, 1610 nm, Soil and plant moisture
    "swir22",     # B12: SWIR 2, 20 m/px, 2190 nm, Geology and burn scars
]

GROUPS = {
    "Visible 1":  ["true_colour", "coastal"],
    "Visible 2":  ["blue", "green", "red"],
    "Red edge": ["rededge1", "rededge2", "rededge3"],
    "NIR":      ["nir", "nir08", "nir09"],
    "SWIR":     ["swir16", "swir22"],
}

LABELS = {
    "true_colour": "True Colour: RGB Composite\n(10 m/px)\nNatural visualization",
    "coastal": "B1: Coastal Aerosol\n(60 m/px, 443 nm)\nAtmospheric correction",
    "blue": "B2: Blue\n(10 m/px, 490 nm)\nForest type mapping",
    "green": "B3: Green\n(10 m/px, 560 nm)\nWater body analysis",
    "red": "B4: Red\n(10 m/px, 665 nm)\nVegetation discrimination",
    "rededge1": "B5: Red Edge 1\n(20 m/px, 705 nm)\nVegetation stress detection",
    "rededge2": "B6: Red Edge 2\n(20 m/px, 740 nm)\nLeaf area index monitoring",
    "rededge3": "B7: Red Edge 3\n(20 m/px, 783 nm)\nBiomass estimation",
    "nir": "B8: NIR\n(10 m/px, 842 nm)\nShorelines and biomass",
    "nir08": "B8A: Narrow NIR\n(20 m/px, 865 nm)\nVegetation classification",
    "nir09": "B9: Water Vapour\n(60 m/px, 945 nm)\nAtmospheric water vapour",
    "swir16": "B11: SWIR 1\n(20 m/px, 1610 nm)\nSoil and plant moisture",
    "swir22": "B12: SWIR 2\n(20 m/px, 2190 nm)\nGeology and burn scars",
}

In [5]:
longitude = 5.36
latitude = 44.73
size_meters = 4000.0
max_cloud = 10
collection = "sentinel-2-l2a",
scale_factor = 10_000.0  # For S2
query = None

# Forest fires in June-July 2026

# Before the forest fire (in August, one year before)
date_start_before = "2025-08-01"
date_end_before = "2025-08-31"

# After the forest fire (in August, immediately after)
date_start_after = "2026-08-01"
date_end_ater = "2026-08-31"

In [6]:
# Before
img_before, date_before = fetch_s2(
    longitude,
    latitude,
    size_meters,
    date_start_before,
    date_end_before,    
    bands=BANDS,
    stac_url=STAC_URL,
    collection=collection,
    max_cloud=max_cloud,
    query=query,
    scale_factor=scale_factor,
)
print(f"Date before: {date_before}")

Date before: 2025-08-08


In [ ]:
# After
img_after, date_after = fetch_s2(
    longitude,
    latitude,
    size_meters,
    date_start_after,
    date_end_ater,    
    bands=BANDS,
    stac_url=STAC_URL,
    collection=collection,
    max_cloud=max_cloud,
    query=query,
    scale_factor=scale_factor,
)
print(f"Date after: {date_after}")

In [ ]:
LO_PCT = 2
HI_PCT = 98  
idx = {name: i for i, name in enumerate(BANDS)}

# 13 bands/rows, 3 columns: Before, After, Difference
nrows = len(BANDS)
fig, axes = plt.subplots(nrows, 3, figsize=(9, 3.2 * nrows), squeeze=False)

for ax in axes.ravel():
    ax.axis("off")

for r, name in enumerate(BANDS):
    
    b_idx = idx[name]    
    img_b = img_before[b_idx]
    img_a = img_after[b_idx]
    
    # Compute shared bounds across both dates for valid comparative visualization
    lo, hi = np.percentile(np.concatenate([img_b, img_a]), (LO_PCT, HI_PCT))

    # Before
    axes[r, 0].imshow(softclip(img_b, lo=lo, hi=hi), cmap="gray")
    axes[r, 0].set_title(f"{LABELS[name]}\n[Before: {date_before}]", fontsize=9)

    # After
    axes[r, 1].imshow(softclip(img_a, lo=lo, hi=hi), cmap="gray")
    axes[r, 1].set_title(f"{LABELS[name]}\n[After: {date_after}]", fontsize=9)

    # Signed Difference (Blue-White-Red, symmetric around 0)    
    diff = img_a - img_b
    max_abs_diff = max(float(np.percentile(np.abs(diff), 98)), 1e-9)    
    im_diff = axes[r, 2].imshow(diff, cmap="coolwarm", vmin=-max_abs_diff, vmax=max_abs_diff)
    axes[r, 2].set_title(f"{LABELS[name]}\n[Difference (After - Before)]", fontsize=9)
    
    fig.colorbar(im_diff, ax=axes[r, 2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()